# ASTER — Simulation Benchmark

Applies ASTER to 3 embryo sections × 6 degradation scenarios.
Outputs imputed `.h5ad` files and MSE / NRMSE summary CSV.

## Data you need to download first

This repository ships the data directories **empty**. Download package
**`<SIMULATION_URL>`** and unpack it into `preprocess_data/simulation/` (see the README
there): 3 files, ~1.1 GB -- `embryo_section{1,2,3}_processed.h5ad`. These are the
benchmark-ready inputs, so no upstream preprocessing is needed; each file carries the
degradation scenarios as layers, which `SCENARIOS` below selects.

Check your download before running the rest:

In [ ]:
import subprocess, sys

from repro_st_aster.common import find_repo_root

REPO_ROOT = find_repo_root()
subprocess.run([sys.executable, str(REPO_ROOT / 'scripts' / 'check_data.py'), 'simulation'])

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
import matplotlib.pyplot as plt
from tqdm import tqdm

from repro_st_aster.aster_ntd import ASTER, load_simulation_slice
from repro_st_aster.common import find_repo_root


In [ ]:
REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / 'preprocess_data' / 'simulation'
OUTPUT_DIR = REPO_ROOT / 'results' / 'simulation_notebook'

RANK = 128
LR = 0.001
MAX_EPOCH = 1500

SECTIONS = ['embryo_section1', 'embryo_section2', 'embryo_section3']
SCENARIOS = ['sr_norm_0.3', 'sr_norm_0.4', 'sr_norm_0.5', 'noise_norm_sd_0.5', 'noise_norm_sd_1.0', 'noise_norm_sd_2.0']
VIZ_GENES = ['Gm42418', 'Rpl41']

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def compute_metrics(adata, imputed_layer: str) -> dict:
    """MSE / NRMSE on top-100 HVGs and per-gene metrics for VIZ_GENES."""
    y_true = adata.layers['original_norm']
    y_pred = adata.layers[imputed_layer]
    if sp.issparse(y_true): y_true = y_true.toarray()
    if sp.issparse(y_pred): y_pred = y_pred.toarray()

    tmp = adata.copy()
    tmp.X = tmp.layers['original_norm']
    sc.pp.highly_variable_genes(tmp, n_top_genes=100, flavor='seurat_v3')
    hvg_idx = np.where(tmp.var['highly_variable'].values)[0]

    mse_hvg  = float(np.mean((y_true[:, hvg_idx] - y_pred[:, hvg_idx]) ** 2))
    rmse_hvg = float(np.sqrt(mse_hvg))
    nrmse    = rmse_hvg / float(np.std(y_true[:, hvg_idx]))
    mse_all  = float(np.mean((y_true - y_pred) ** 2))

    gene_metrics = {}
    for gene in VIZ_GENES:
        matches = [g for g in adata.var_names if g.lower() == gene.lower()]
        if not matches:
            continue
        gname = matches[0]
        gi    = adata.var_names.get_loc(gname)
        if not isinstance(gi, (int, np.integer)):
            gi = gi[0]
        t, p = y_true[:, gi], y_pred[:, gi]
        corr = 0.0 if t.std() == 0 or p.std() == 0 else float(np.corrcoef(t, p)[0, 1])
        gene_metrics[gname] = {'mse': float(np.mean((t - p) ** 2)), 'corr': corr}

    return dict(mse_hvg=mse_hvg, rmse_hvg=rmse_hvg, nrmse=nrmse, mse_all=mse_all,
                gene_metrics=gene_metrics)

In [ ]:
def visualize(adata, gene: str, input_layer: str, imputed_layer: str, save_path: str):
    """Three-panel: Original | Input | ASTER for one gene."""
    matches = [g for g in adata.var_names if g.lower() == gene.lower()]
    if not matches:
        return
    gname = matches[0]
    gi    = adata.var_names.get_loc(gname)
    if not isinstance(gi, (int, np.integer)):
        gi = gi[0]

    xc = adata.obs['array_col'].values
    yc = adata.obs['array_row'].values

    def _dense(layer_key):
        d = adata.layers[layer_key]
        return d.toarray() if sp.issparse(d) else d

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, (title, lkey) in zip(axes, [
        ('Original',        'original_norm'),
        (f'Input ({input_layer})', input_layer),
        ('ASTER',           imputed_layer),
    ]):
        ax.scatter(xc, yc, c=_dense(lkey)[:, gi], cmap='RdYlBu_r', s=5)
        ax.set_title(title, fontsize=14)
        ax.invert_yaxis(); ax.set_aspect('equal'); ax.axis('off')

    plt.suptitle(f'Gene: {gname}', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

In [ ]:
all_metrics = []

for section in tqdm(SECTIONS, desc='Sections'):
    h5ad_path = DATA_DIR / f'{section}_processed.h5ad'
    if not h5ad_path.exists():
        print(f'Missing: {h5ad_path}'); continue

    adata_full = sc.read_h5ad(h5ad_path)

    for scenario in tqdm(SCENARIOS, desc=section, leave=False):
        try:
            # ── 1. Load ──────────────────────────────────────────────────────
            data = load_simulation_slice(h5ad_path, scenario)

            # ── 2. Fit ───────────────────────────────────────────────────────
            model = ASTER(
                expr_tensor    = data['expr_tensor'],
                gene_names     = data['gene_names'],
                n_x            = data['n_x'],
                n_y            = data['n_y'],
                coords_mapping = data['coords_mapping'],
            )
            model.fit(rank_x=RANK, rank_y=RANK, rank_g=RANK,
                      lr=LR, max_epoch=MAX_EPOCH,
                      save_dir=str(OUTPUT_DIR / section / scenario), verbose=True)

            # ── 3. Store imputed layer ────────────────────────────────────────
            expr_imputed, _ = model.get_imputed_matrix()
            imputed_key = f'ASTER_{scenario}'
            adata_full.layers[imputed_key] = sp.csr_matrix(expr_imputed)

            # ── 4. Metrics ───────────────────────────────────────────────────
            m = compute_metrics(adata_full, imputed_key)
            row = dict(section=section, scenario=scenario,
                       mse_hvg=m['mse_hvg'], nrmse=m['nrmse'], mse_all=m['mse_all'])
            for gname, gm in m['gene_metrics'].items():
                row[f'{gname}_mse']  = gm['mse']
                row[f'{gname}_corr'] = gm['corr']
            all_metrics.append(row)

            # ── 5. Visualize ─────────────────────────────────────────────────
            for gene in VIZ_GENES:
                visualize(
                    adata_full, gene, scenario, imputed_key,
                    save_path=OUTPUT_DIR / f'{section}_{scenario}_{gene}.png',
                )

            model.clear_gpu()

        except Exception as e:
            import traceback; traceback.print_exc()
            all_metrics.append(dict(section=section, scenario=scenario,
                                    mse_hvg=np.nan, nrmse=np.nan, status=str(e)))

    # save per-section h5ad with all imputed layers
    out_h5ad = OUTPUT_DIR / f'{section}_ASTER_all_scenarios.h5ad'
    adata_full.write_h5ad(out_h5ad)
    print(f'Saved: {out_h5ad}')

In [ ]:
df = pd.DataFrame(all_metrics)
csv_path = OUTPUT_DIR / 'Simulation_ASTER_metrics.csv'
df.to_csv(csv_path, index=False)

print(df[['section', 'scenario', 'mse_hvg', 'nrmse']].to_string(index=False))
print(f'\nSaved: {csv_path}')